# FogoCerto · Visão: treino e avaliação do detector de fumaça e fogo

**Global Solution FIAP 2026.2 · CitizenTech · Applied Computer Vision**
Integrantes: Breno Silva (RM99275), Eduardo Araujo (RM99758), Gabriela Trevisan (RM99500), Gustavo Akio (RM550241), Rafael Franck (RM550875).

Este notebook treina um **YOLO11** (detecção de objetos) para encontrar **fumaça** e **fogo** em imagens e mede o resultado de duas formas:

1. **Por caixa** (métrica padrão de detecção): mAP@50, mAP@50-95, precisão, recall e matriz de confusão no conjunto de **teste**, que o modelo nunca viu.
2. **Por imagem** (a visão do produto): a foto tem fogo, fumaça ou nada? Daqui sai o **limiar de confiança** da API, escolhido por um critério explícito.

Ele também separa os **erros reais** (falsos alarmes e fogos perdidos) para o relatório e testa as **imagens próprias** do grupo.

**Como rodar no Colab:** `Ambiente de execução > Alterar o tipo de ambiente de execução > GPU T4`, depois `Executar tudo`. Tudo é salvo no Google Drive (`MyDrive/FogoCerto_visao`). Se a sessão cair, rode tudo de novo: o treino **continua de onde parou**.

**Dataset:** D-Fire (Venâncio, Lisboa e Barbosa, UFMG, 2022), licença **CC0 1.0**, mais de 21 mil imagens com rótulos YOLO. Versão do Kaggle já dividida em treino, validação e teste: `sayedgamal99/smoke-fire-detection-yolo`.

## 1. Configuração

In [ ]:
import os, sys, json, time, shutil, random, subprocess, csv
from pathlib import Path
from collections import Counter

EM_COLAB = "google.colab" in sys.modules

# Hiperparâmetros (as variáveis FC_* só servem para o teste local rápido)
MODELO_BASE   = os.environ.get("FC_MODELO", "yolo11n.pt")  # n = leve, roda em CPU; yolo11s.pt se sobrar GPU
EPOCAS        = int(os.environ.get("FC_EPOCAS", 30))
TAM_IMG       = int(os.environ.get("FC_TAM", 640))
LOTE          = int(os.environ.get("FC_LOTE", 32))
FRACAO        = float(os.environ.get("FC_FRACAO", 1.0))  # 0.5 = usa metade do treino (corta o tempo)
PACIENCIA     = 8      # para cedo se a validação não melhorar em 8 épocas
SEMENTE       = 42
NOME_EXECUCAO = os.environ.get("FC_NOME", "yolo11n_dfire")
KAGGLE_DATASET = "sayedgamal99/smoke-fire-detection-yolo"
META_SENSIBILIDADE_DUVIDA = 0.95  # ver seção 6

if EM_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA_SAIDA = Path("/content/drive/MyDrive/FogoCerto_visao")
else:
    PASTA_SAIDA = Path(os.environ.get("FC_SAIDA", "saida_visao")).resolve()

PASTA_RUNS = PASTA_SAIDA / "runs"
PASTA_RESULTADOS = PASTA_SAIDA / "resultados"   # copiar para visao/resultados/ no repositório
for p in (PASTA_SAIDA, PASTA_RUNS, PASTA_RESULTADOS):
    p.mkdir(parents=True, exist_ok=True)
random.seed(SEMENTE)
print("Saída em:", PASTA_SAIDA)

In [ ]:
if EM_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3,<9", "kagglehub"], check=True)

import numpy as np
import torch
import ultralytics
from ultralytics import YOLO
import matplotlib
if not EM_COLAB and "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

GPU = torch.cuda.is_available()
DISPOSITIVO = 0 if GPU else "cpu"
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__,
      "| GPU:", torch.cuda.get_device_name(0) if GPU else "nenhuma")
if EM_COLAB and not GPU:
    raise SystemExit("Sem GPU. Ative em: Ambiente de execução > Alterar o tipo > GPU T4.")


def mostrar(caminho):
    # Exibe uma imagem salva (no Colab) ou só informa onde ela está (fora do notebook).
    caminho = Path(caminho)
    if not caminho.exists():
        print("(não encontrado)", caminho)
        return
    try:
        from IPython.display import Image as IPImage, display
        if "ipykernel" in sys.modules:
            display(IPImage(filename=str(caminho)))
            return
    except ImportError:
        pass
    print("figura:", caminho)

## 2. Dataset

Baixa o D-Fire do Kaggle (3,1 GB). Se o Kaggle pedir login, crie um token em *kaggle.com > Settings > API* e cadastre `KAGGLE_USERNAME` e `KAGGLE_KEY` nos **Secrets** do Colab (ícone de chave na barra lateral). Para usar uma cópia já baixada, defina a variável de ambiente `DFIRE_DIR`.

Os nomes das classes são trocados para português (`0 = fumaca`, `1 = fogo`), que é o que a API devolve.

In [ ]:
def achar_raiz(base):
    # Pasta que contém train/images (no Kaggle fica em <download>/data).
    base = Path(base)
    if (base / "train" / "images").is_dir():
        return base
    for p in sorted(base.rglob("images")):
        if p.parent.name == "train" and p.is_dir():
            return p.parent.parent
    raise FileNotFoundError(f"não achei train/images dentro de {base}")


if os.environ.get("DFIRE_DIR"):
    RAIZ = achar_raiz(os.environ["DFIRE_DIR"])
else:
    import kagglehub
    if EM_COLAB:
        try:  # usa os Secrets do Colab, se existirem
            from google.colab import userdata
            for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
                os.environ.setdefault(k, userdata.get(k))
        except Exception:
            pass
    try:
        RAIZ = achar_raiz(kagglehub.dataset_download(KAGGLE_DATASET))
    except Exception as erro:
        raise SystemExit(f"Falha ao baixar do Kaggle ({erro}). Veja a instrução acima sobre KAGGLE_USERNAME/KAGGLE_KEY.")

SPLIT_VAL = "val" if (RAIZ / "val" / "images").is_dir() else "valid"
DATA_YAML = PASTA_SAIDA / "dfire_fogocerto.yaml"
DATA_YAML.write_text(
    f"path: {RAIZ.as_posix()}\n"
    f"train: train/images\nval: {SPLIT_VAL}/images\ntest: test/images\n"
    "names:\n  0: fumaca\n  1: fogo\n",
    encoding="utf-8",
)
print("Dataset em", RAIZ)
print(DATA_YAML.read_text())

### 2.1 O que tem no dataset

In [ ]:
EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
NOMES = {0: "fumaca", 1: "fogo"}


def imagens(split):
    return sorted(p for p in (RAIZ / split / "images").iterdir() if p.suffix.lower() in EXT)


def rotulos(img):
    # Caixas reais (classe, xc, yc, w, h normalizados). Arquivo vazio ou ausente = imagem sem fogo nem fumaça.
    lab = img.parent.parent / "labels" / (img.stem + ".txt")
    if not lab.exists():
        return []
    caixas = []
    for linha in lab.read_text().splitlines():
        partes = linha.split()
        if len(partes) >= 5:
            caixas.append((int(partes[0]), *map(float, partes[1:5])))
    return caixas


def classe_real(caixas):
    # Mesma regra do produto (visao/src/decisao.py): fogo tem prioridade sobre fumaça.
    ids = {c for c, *_ in caixas}
    return "fogo" if 1 in ids else "fumaca" if 0 in ids else "nada"


def categoria(caixas):
    ids = {c for c, *_ in caixas}
    return {frozenset(): "nenhum", frozenset({0}): "so_fumaca", frozenset({1}): "so_fogo"}.get(frozenset(ids), "fogo_e_fumaca")


ESTATISTICAS = {}
for split in ("train", SPLIT_VAL, "test"):
    cats, caixas_por_classe = Counter(), Counter()
    for img in imagens(split):
        cx = rotulos(img)
        cats[categoria(cx)] += 1
        caixas_por_classe.update(NOMES[c] for c, *_ in cx)
    ESTATISTICAS[split] = {"imagens": sum(cats.values()), **cats, **{f"caixas_{k}": v for k, v in caixas_por_classe.items()}}

colunas = ["imagens", "so_fumaca", "so_fogo", "fogo_e_fumaca", "nenhum", "caixas_fumaca", "caixas_fogo"]
print(f"{'split':8}" + "".join(f"{c:>15}" for c in colunas))
for s, e in ESTATISTICAS.items():
    print(f"{s:8}" + "".join(f"{e.get(c, 0):>15}" for c in colunas))
(PASTA_RESULTADOS / "estatisticas_dataset.json").write_text(json.dumps(ESTATISTICAS, indent=2), encoding="utf-8")

In [ ]:
def desenhar(ax, img, caixas_norm=(), caixas_px=(), titulo=""):
    # caixas_norm: rótulos reais (verde); caixas_px: (classe, conf, x1, y1, x2, y2) previstas.
    im = Image.open(img).convert("RGB")
    W, H = im.size
    ax.imshow(im)
    for c, xc, yc, w, h in caixas_norm:
        ax.add_patch(patches.Rectangle(((xc - w / 2) * W, (yc - h / 2) * H), w * W, h * H,
                                       fill=False, ec="lime", lw=2))
    for nome, conf, x1, y1, x2, y2 in caixas_px:
        cor = "red" if nome == "fogo" else "deepskyblue"
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=cor, lw=2))
        ax.text(x1, y1, f"{nome} {conf:.2f}", color="white", fontsize=7,
                bbox=dict(facecolor=cor, alpha=.7, pad=1, lw=0))
    ax.set_title(titulo, fontsize=8)
    ax.axis("off")


def grade(lista, caminho, titulo, colunas=4):
    # lista de dicts com chaves img, norm, px, titulo
    if not lista:
        print("nada para mostrar em", caminho.name)
        return
    linhas = (len(lista) + colunas - 1) // colunas
    fig, axs = plt.subplots(linhas, colunas, figsize=(4 * colunas, 3.3 * linhas), squeeze=False)
    for ax in axs.flat:
        ax.axis("off")
    for ax, item in zip(axs.flat, lista):
        desenhar(ax, item["img"], item.get("norm", ()), item.get("px", ()), item.get("titulo", ""))
    fig.suptitle(titulo)
    fig.tight_layout()
    fig.savefig(caminho, dpi=110)
    plt.show()
    plt.close(fig)


treino = imagens("train")
amostra = [p for p in random.sample(treino, min(400, len(treino)))]
com_rotulo = [p for p in amostra if rotulos(p)][:8]
sem_rotulo = [p for p in amostra if not rotulos(p)][:4]
grade([{"img": p, "norm": rotulos(p), "titulo": f"{p.name} · {classe_real(rotulos(p))}"} for p in com_rotulo + sem_rotulo],
      PASTA_RESULTADOS / "exemplos_dataset.png", "Exemplos do D-Fire (verde = rótulo real)")

## 3. Treino (transfer learning)

Parte de pesos pré-treinados no COCO e ajusta para as 2 classes. O Ultralytics já aplica *data augmentation* (mosaico, HSV, giro horizontal, escala). Cada época é salva no Drive; se a sessão cair, esta célula **retoma** do `last.pt`.

Tempo esperado numa T4: cerca de 3 a 5 minutos por época com o dataset inteiro. Para caber no tempo, use `FRACAO = 0.5` ou menos épocas.

In [ ]:
def achar_peso(nome):
    achados = sorted(PASTA_RUNS.rglob(f"{NOME_EXECUCAO}/weights/{nome}"))
    return achados[0] if achados else None


def treino_concluido(peso):
    # Ao terminar, o Ultralytics remove o otimizador do checkpoint e marca epoch = -1.
    # Um "resume" nesse caso começaria um treino NOVO por cima, por isso checamos antes.
    ck = torch.load(peso, map_location="cpu", weights_only=False)
    return ck.get("optimizer") is None or ck.get("epoch", -1) < 0


inicio = time.time()
ultimo = achar_peso("last.pt")
if ultimo and treino_concluido(ultimo):
    print("Treino já concluído; usando os pesos salvos em", ultimo.parent)
elif ultimo:
    print("Retomando de", ultimo)
    YOLO(str(ultimo)).train(resume=True)
else:
    YOLO(MODELO_BASE).train(
        data=str(DATA_YAML), epochs=EPOCAS, imgsz=TAM_IMG, batch=LOTE, fraction=FRACAO,
        patience=PACIENCIA, seed=SEMENTE, deterministic=True, device=DISPOSITIVO,
        workers=2, project=str(PASTA_RUNS), name=NOME_EXECUCAO, exist_ok=True, plots=True,
    )
print(f"tempo nesta sessão: {(time.time() - inicio) / 60:.1f} min")

PESO = achar_peso("best.pt")
PASTA_TREINO = PESO.parent.parent
print("melhor peso:", PESO)
for arq in ("results.png", "results.csv", "args.yaml"):
    if (PASTA_TREINO / arq).exists():
        shutil.copy(PASTA_TREINO / arq, PASTA_RESULTADOS / f"treino_{arq}")
mostrar(PASTA_TREINO / "results.png")

## 4. Avaliação por caixa no conjunto de teste

Métricas padrão de detecção, calculadas pelo Ultralytics no split **test** (4.306 imagens que não entraram no treino nem na escolha do melhor peso).

- **mAP@50**: área sob a curva precisão × recall, contando como acerto a caixa que cobre pelo menos 50% (IoU) da caixa real.
- **mAP@50-95**: média do mAP para IoU de 0,50 a 0,95 (exige caixa mais justa).
- A matriz de confusão tem a linha e a coluna **background**: caixa prevista onde não há nada (falso alarme) ou caixa real não encontrada.

In [ ]:
modelo = YOLO(str(PESO))
m = modelo.val(data=str(DATA_YAML), split="test", imgsz=TAM_IMG, batch=LOTE, device=DISPOSITIVO,
               plots=True, project=str(PASTA_RUNS), name=f"{NOME_EXECUCAO}_teste", exist_ok=True)

por_classe = {}
for i, c in enumerate(m.box.ap_class_index):
    por_classe[m.names[int(c)]] = {
        "precisao": round(float(m.box.p[i]), 4), "recall": round(float(m.box.r[i]), 4),
        "mAP50": round(float(m.box.ap50[i]), 4), "mAP50_95": round(float(m.box.ap[i]), 4),
    }
METRICAS_CAIXA = {
    "modelo_base": MODELO_BASE, "epocas_configuradas": EPOCAS, "tam_img": TAM_IMG, "fracao_treino": FRACAO,
    "imagens_teste": ESTATISTICAS["test"]["imagens"],
    "mAP50": round(float(m.box.map50), 4), "mAP50_95": round(float(m.box.map), 4),
    "precisao": round(float(m.box.mp), 4), "recall": round(float(m.box.mr), 4),
    "por_classe": por_classe,
    "velocidade_ms_por_imagem": {k: round(v, 2) for k, v in m.speed.items()},
    "dispositivo": torch.cuda.get_device_name(0) if GPU else "cpu",
}
(PASTA_RESULTADOS / "metricas_teste_caixas.json").write_text(json.dumps(METRICAS_CAIXA, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(METRICAS_CAIXA, indent=2, ensure_ascii=False))

pasta_val = Path(m.save_dir)
destino = PASTA_RESULTADOS / "teste_caixas"
destino.mkdir(exist_ok=True)
for png in pasta_val.glob("*.png"):
    if "batch" not in png.name:  # os mosaicos de batch ficam só no Drive
        shutil.copy(png, destino / png.name)
for nome in ("confusion_matrix_normalized.png", "confusion_matrix.png"):
    mostrar(pasta_val / nome)

## 5. Avaliação por imagem (a visão do produto)

A API responde uma coisa por foto: **fogo**, **fumaça** ou **nada**. A regra é a mesma de `visao/src/decisao.py`:
se há caixa de fogo com confiança ≥ limiar → *fogo*; senão, caixa de fumaça ≥ limiar → *fumaça*; senão → *nada*.
O rótulo real da imagem segue a mesma prioridade (tem caixa de fogo → *fogo*; só fumaça → *fumaça*; sem caixa → *nada*).

Rodamos o modelo **uma vez** em todo o teste com confiança mínima de 0,05 e depois variamos o limiar sem rodar de novo.

In [ ]:
CL = ["fogo", "fumaca", "nada"]
imgs_teste = imagens("test")
REAL = {p.name: classe_real(rotulos(p)) for p in imgs_teste}
CAMINHO = {p.name: p for p in imgs_teste}
PREV = {}  # nome -> [(classe, conf, x1, y1, x2, y2)]
for r in modelo.predict(source=str(RAIZ / "test" / "images"), stream=True, conf=0.05, imgsz=TAM_IMG,
                        device=DISPOSITIVO, verbose=False):
    caixas = r.boxes
    PREV[Path(r.path).name] = [
        (modelo.names[int(c)], float(cf), *map(float, xyxy))
        for c, cf, xyxy in zip(caixas.cls.tolist(), caixas.conf.tolist(), caixas.xyxy.tolist())
    ]
print("imagens avaliadas:", len(PREV), "de", len(REAL))


def classe_prevista(dets, limiar):
    if any(d[0] == "fogo" and d[1] >= limiar for d in dets):
        return "fogo"
    if any(d[0] == "fumaca" and d[1] >= limiar for d in dets):
        return "fumaca"
    return "nada"


def matriz(limiar):
    M = np.zeros((3, 3), dtype=int)  # linhas = real, colunas = previsto
    for nome, real in REAL.items():
        M[CL.index(real), CL.index(classe_prevista(PREV.get(nome, []), limiar))] += 1
    return M


def div(a, b):
    return float(a) / float(b) if b else 0.0


def resumo(M, limiar):
    r = {"limiar": round(limiar, 2), "acuracia": div(np.trace(M), M.sum())}
    for i, c in enumerate(CL):
        p, rc = div(M[i, i], M[:, i].sum()), div(M[i, i], M[i, :].sum())
        r[f"precisao_{c}"], r[f"recall_{c}"], r[f"f1_{c}"] = p, rc, div(2 * p * rc, p + rc)
    # "ocorrência" = a foto tem fogo ou fumaça (qualquer um dos dois), contra "nada"
    vp, fn = M[:2, :2].sum(), M[:2, 2].sum()
    fp, vn = M[2, :2].sum(), M[2, 2]
    r["sensibilidade_ocorrencia"] = div(vp, vp + fn)      # das fotos com fogo/fumaça, quantas avisamos
    r["falso_alarme"] = div(fp, fp + vn)                  # das fotos sem nada, quantas alarmamos
    r["precisao_ocorrencia"] = div(vp, vp + fp)
    r["f1_ocorrencia"] = div(2 * vp, 2 * vp + fp + fn)
    r["f1_macro"] = float(np.mean([r[f"f1_{c}"] for c in CL]))
    return {k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}


GRADE = [round(x, 2) for x in np.arange(0.05, 0.81, 0.05)]
TABELA = [resumo(matriz(t), t) for t in GRADE]
with open(PASTA_RESULTADOS / "tabela_limiares.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(TABELA[0]))
    w.writeheader()
    w.writerows(TABELA)
print(f"{'limiar':>7}{'sensib.':>9}{'f.alarme':>10}{'F1 ocorr.':>11}{'F1 macro':>10}")
for l in TABELA:
    print(f"{l['limiar']:>7}{l['sensibilidade_ocorrencia']:>9.3f}{l['falso_alarme']:>10.3f}{l['f1_ocorrencia']:>11.3f}{l['f1_macro']:>10.3f}")

## 6. Escolha do limiar (com critério)

- **`limiar` (confirmar)**: o que dá o **maior F1 de ocorrência** no teste, isto é, o melhor equilíbrio entre não perder fogo e não alarmar à toa. Acima dele a API diz "possível fogo/fumaça".
- **`limiar_duvida`**: o maior limiar abaixo do anterior que ainda avisa **pelo menos 95%** das fotos com fogo ou fumaça. Entre os dois, a API responde **com dúvida** ("não foi possível confirmar; se houver perigo, procure a defesa civil"). Assim o produto não esconde um caso provável, mas também não afirma o que não tem certeza.

> Observação honesta: o limiar é escolhido no próprio conjunto de teste. Para um número 100% sem viés, a escolha deveria ser feita na validação; com o tempo da GS, registramos isso como limitação no relatório.

In [ ]:
melhor = max(TABELA, key=lambda l: (l["f1_ocorrencia"], l["limiar"]))
LIMIAR = melhor["limiar"]
candidatos = [l["limiar"] for l in TABELA if l["limiar"] < LIMIAR and l["sensibilidade_ocorrencia"] >= META_SENSIBILIDADE_DUVIDA]
LIMIAR_DUVIDA = max(candidatos) if candidatos else GRADE[0]
LIMIAR_DUVIDA = round(max(0.05, min(LIMIAR_DUVIDA, LIMIAR - 0.05)), 2)
duvida = next(l for l in TABELA if l["limiar"] == LIMIAR_DUVIDA)

ESCOLHA = {
    "limiar": LIMIAR, "limiar_duvida": LIMIAR_DUVIDA,
    "criterio": "limiar = maior F1 de ocorrência no teste; limiar_duvida = maior limiar abaixo dele com "
                f"sensibilidade >= {META_SENSIBILIDADE_DUVIDA}",
    "no_limiar": melhor, "no_limiar_duvida": duvida,
}
(PASTA_RESULTADOS / "limiar_recomendado.json").write_text(json.dumps(ESCOLHA, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(ESCOLHA, indent=2, ensure_ascii=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(GRADE, [l["sensibilidade_ocorrencia"] for l in TABELA], label="sensibilidade (fotos com fogo/fumaça avisadas)")
ax.plot(GRADE, [l["falso_alarme"] for l in TABELA], label="falso alarme (fotos sem nada alarmadas)")
ax.plot(GRADE, [l["f1_ocorrencia"] for l in TABELA], label="F1 de ocorrência")
ax.axvline(LIMIAR, color="k", ls="--", lw=1)
ax.axvline(LIMIAR_DUVIDA, color="gray", ls=":", lw=1)
ax.text(LIMIAR, 0.02, f" limiar {LIMIAR}", fontsize=8)
ax.text(LIMIAR_DUVIDA, 0.10, f" dúvida {LIMIAR_DUVIDA}", fontsize=8, color="gray")
ax.set_xlabel("limiar de confiança")
ax.set_ylim(0, 1.02)
ax.legend(fontsize=8, loc="center right")
ax.set_title("Avaliação por imagem no teste")
fig.tight_layout()
fig.savefig(PASTA_RESULTADOS / "curva_limiares.png", dpi=120)
plt.show()
plt.close(fig)

M = matriz(LIMIAR)
fig, ax = plt.subplots(figsize=(5, 4.2))
pct = M / M.sum(axis=1, keepdims=True).clip(min=1)
ax.imshow(pct, cmap="Oranges", vmin=0, vmax=1)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{M[i, j]}\n{pct[i, j]:.0%}", ha="center", va="center",
                color="white" if pct[i, j] > .6 else "black", fontsize=10)
ax.set_xticks(range(3), CL)
ax.set_yticks(range(3), CL)
ax.set_xlabel("previsto pelo modelo")
ax.set_ylabel("real (rótulo)")
ax.set_title(f"Matriz de confusão por imagem (limiar {LIMIAR})")
fig.tight_layout()
fig.savefig(PASTA_RESULTADOS / "matriz_confusao_por_imagem.png", dpi=120)
plt.show()
plt.close(fig)
(PASTA_RESULTADOS / "matriz_confusao_por_imagem.json").write_text(
    json.dumps({"classes": CL, "linhas_real_colunas_previsto": M.tolist(), "limiar": LIMIAR}, indent=2), encoding="utf-8")

## 7. Erros reais (para o relatório)

Separamos as imagens em que o modelo errou, no limiar escolhido:

- **Falso alarme**: foto sem fogo nem fumaça que o modelo marcou (esperado: nuvem, neblina, poeira, pôr do sol, luz forte, reflexo).
- **Fogo/fumaça perdidos**: foto com rótulo que o modelo deixou passar (esperado: fumaça fina ou distante, fogo pequeno).
- **Troca de classe**: fumaça vista como fogo, ou o contrário.

O arquivo `erros_para_anotar.csv` lista os casos com uma coluna `causa_provavel` **em branco**: olhem as imagens e preencham com o que de fato aparece. É essa tabela que mostra honestidade no relatório.

In [ ]:
def maior_conf(nome):
    return max((d[1] for d in PREV.get(nome, [])), default=0.0)


def item(nome, titulo):
    return {"img": CAMINHO[nome], "norm": rotulos(CAMINHO[nome]),
            "px": [d for d in PREV.get(nome, []) if d[1] >= LIMIAR_DUVIDA], "titulo": titulo}


prev = {n: classe_prevista(PREV.get(n, []), LIMIAR) for n in REAL}
falso_alarme = sorted([n for n in REAL if REAL[n] == "nada" and prev[n] != "nada"], key=maior_conf, reverse=True)
perdidos = sorted([n for n in REAL if REAL[n] != "nada" and prev[n] == "nada"], key=maior_conf)
trocas = [n for n in REAL if {REAL[n], prev[n]} == {"fogo", "fumaca"}]
print(f"falsos alarmes: {len(falso_alarme)} | perdidos: {len(perdidos)} | trocas fogo/fumaça: {len(trocas)}")

grade([item(n, f"{n} · previsto {prev[n]} ({maior_conf(n):.2f})") for n in falso_alarme[:12]],
      PASTA_RESULTADOS / "erros_falso_alarme.png", "Falsos alarmes (sem rótulo real; vermelho/azul = previsto)")
grade([item(n, f"{n} · real {REAL[n]}") for n in perdidos[:12]],
      PASTA_RESULTADOS / "erros_perdidos.png", "Fogo/fumaça perdidos (verde = rótulo real)")
grade([item(n, f"{n} · real {REAL[n]} → previsto {prev[n]}") for n in trocas[:8]],
      PASTA_RESULTADOS / "erros_troca_classe.png", "Troca entre fogo e fumaça")

with open(PASTA_RESULTADOS / "erros_para_anotar.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["tipo", "arquivo", "real", "previsto", "maior_confianca", "causa_provavel"])
    for tipo, lista in (("falso_alarme", falso_alarme[:40]), ("perdido", perdidos[:40]), ("troca", trocas[:20])):
        for n in lista:
            w.writerow([tipo, n, REAL[n], prev[n], f"{maior_conf(n):.3f}", ""])

## 8. Imagens próprias do grupo (teste fora do D-Fire)

Coloque fotos tiradas pelo grupo (ou de fonte com licença livre, anotando a fonte) em `MyDrive/FogoCerto_visao/imagens_proprias/`, com o nome começando pela resposta certa:

- `fogo_...jpg`, `fumaca_...jpg` e `nada_...jpg`
- Inclua casos difíceis: `nada_nuvem_01.jpg`, `nada_neblina_01.jpg`, `nada_por_do_sol_01.jpg`, `nada_poeira_01.jpg`, `fumaca_longe_01.jpg` (estilo câmera de torre)…

**LGPD:** sem rosto, placa ou endereço identificável. As imagens próprias **não** vão para o GitHub (só a tabela de resultados).

In [ ]:
PASTA_PROPRIAS = PASTA_SAIDA / "imagens_proprias"
PASTA_PROPRIAS.mkdir(exist_ok=True)
proprias = sorted(p for p in PASTA_PROPRIAS.iterdir() if p.suffix.lower() in EXT)
if not proprias:
    print("Nenhuma imagem própria em", PASTA_PROPRIAS, "- adicione e rode esta célula de novo.")
else:
    linhas, itens = [], []
    for p in proprias:
        real = p.stem.split("_")[0].lower()
        r = modelo.predict(str(p), conf=0.05, imgsz=TAM_IMG, device=DISPOSITIVO, verbose=False)[0]
        dets = [(modelo.names[int(c)], float(cf), *map(float, xy))
                for c, cf, xy in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist(), r.boxes.xyxy.tolist())]
        previsto = classe_prevista(dets, LIMIAR)
        em_duvida = previsto == "nada" and any(d[1] >= LIMIAR_DUVIDA for d in dets)
        conf = max((d[1] for d in dets), default=0.0)
        linhas.append({"arquivo": p.name, "real": real, "previsto": previsto, "duvida": em_duvida,
                       "maior_confianca": round(conf, 3), "acertou": real == previsto})
        itens.append({"img": p, "px": [d for d in dets if d[1] >= LIMIAR_DUVIDA],
                      "titulo": f"{p.name}\nreal {real} · previsto {previsto}{' (dúvida)' if em_duvida else ''}"})
    with open(PASTA_RESULTADOS / "imagens_proprias_resultados.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(linhas[0]))
        w.writeheader()
        w.writerows(linhas)
    acertos = sum(l["acertou"] for l in linhas)
    print(f"acertos: {acertos}/{len(linhas)}")
    for l in linhas:
        print(l)
    grade(itens[:16], PASTA_RESULTADOS / "imagens_proprias.png", "Imagens próprias do grupo")

## 9. Velocidade (dá para ser "tempo real"?)

Mede o tempo por imagem na GPU e na **CPU** (o servidor da API e um notebook comum não têm GPU).

In [ ]:
def medir(dispositivo, n=30):
    amostra = imgs_teste[:n]
    modelo.predict(str(amostra[0]), imgsz=TAM_IMG, device=dispositivo, verbose=False)  # aquecimento
    t0 = time.perf_counter()
    for p in amostra:
        modelo.predict(str(p), imgsz=TAM_IMG, device=dispositivo, verbose=False)
    ms = (time.perf_counter() - t0) / len(amostra) * 1000
    return {"ms_por_imagem": round(ms, 1), "fps": round(1000 / ms, 1)}


VELOCIDADE = {"cpu": medir("cpu")}
if GPU:
    VELOCIDADE["gpu"] = {**medir(0), "placa": torch.cuda.get_device_name(0)}
VELOCIDADE["obs"] = "inclui leitura da imagem, pré e pós-processamento; tam_img=%d" % TAM_IMG
(PASTA_RESULTADOS / "velocidade.json").write_text(json.dumps(VELOCIDADE, indent=2), encoding="utf-8")
print(VELOCIDADE)

## 10. Guardar o modelo e baixar os resultados

- `best.pt` vai para `MyDrive/FogoCerto_visao/modelos/`. **Não** sobe para o GitHub (o `.gitignore` bloqueia `.pt`): compartilhe o link do Drive no `visao/README.md` e coloque o arquivo em `visao/modelos/best.pt` para rodar a API.
- A pasta `resultados/` (JSONs, CSVs e PNGs leves) vai para `visao/resultados/` no repositório. É a evidência do relatório.

In [ ]:
PASTA_MODELOS = PASTA_SAIDA / "modelos"
PASTA_MODELOS.mkdir(exist_ok=True)
shutil.copy(PESO, PASTA_MODELOS / "best.pt")
print("modelo:", PASTA_MODELOS / "best.pt", f"({(PASTA_MODELOS / 'best.pt').stat().st_size / 1e6:.1f} MB)")

zip_resultados = shutil.make_archive(str(PASTA_SAIDA / "resultados_visao"), "zip", PASTA_RESULTADOS)
print("resultados:", zip_resultados)
if EM_COLAB:
    from google.colab import files
    files.download(zip_resultados)
    files.download(str(PASTA_MODELOS / "best.pt"))